# 🔬 Skin Cancer Detection
### Dataset: https://www.kaggle.com/datasets/fanconic/skin-cancer-malignant-vs-benign
### Classes: benign (0), malignant (1)
### Model: MobileNetV2 — Binary Classification with Class Weights

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, zipfile
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

zip_path = '/content/drive/MyDrive/Datasets/skin-cancer-malignant-vs-benign.zip'
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall('/content/skin')
print('Extracted!')

train_dir = test_dir = None
for root, dirs, files in os.walk('/content/skin'):
    dl = [d.lower() for d in dirs]
    if 'benign' in dl and 'malignant' in dl:
        if 'train' in root.lower():                          train_dir = root
        elif 'test' in root.lower() or 'val' in root.lower(): test_dir  = root

print('Train:', train_dir)
print('Test: ', test_dir)
print('Train classes:', os.listdir(train_dir))

# Count images per class
for cls in os.listdir(train_dir):
    n = len(os.listdir(os.path.join(train_dir, cls)))
    print(f'  {cls}: {n} images')

In [ ]:
IMG_SIZE = 224
BATCH    = 32

train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.15,
    horizontal_flip=True,
    vertical_flip=True,
    width_shift_range=0.1,
    height_shift_range=0.1
)
test_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    train_dir, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH, class_mode='binary', shuffle=True
)
test_data = test_gen.flow_from_directory(
    test_dir, target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH, class_mode='binary', shuffle=False
)

print('Classes:', train_data.class_indices)
print('Train:', train_data.samples, '| Test:', test_data.samples)

# Compute class weights to handle imbalance
labels = train_data.classes
weights = compute_class_weight('balanced', classes=np.unique(labels), y=labels)
class_weights = {i: w for i, w in enumerate(weights)}
print('Class weights:', class_weights)

In [ ]:
base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False

inp = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x   = base(inp, training=False)
x   = layers.GlobalAveragePooling2D()(x)
x   = layers.Dense(128, activation='relu')(x)
x   = layers.Dropout(0.3)(x)
out = layers.Dense(1, activation='sigmoid')(x)

model = Model(inp, out)
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
model.summary()

In [ ]:
# Phase 1: Train top layers with class weights
callbacks = [
    EarlyStopping(patience=5, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.5, patience=2, monitor='val_loss', min_lr=1e-6)
]

history1 = model.fit(
    train_data,
    validation_data=test_data,
    epochs=15,
    callbacks=callbacks,
    class_weight=class_weights
)
print(f'Phase 1 Best: {max(history1.history["val_accuracy"])*100:.2f}%')

In [ ]:
# Phase 2: Fine-tune last 30 layers
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

callbacks2 = [
    EarlyStopping(patience=4, restore_best_weights=True, monitor='val_accuracy'),
    ReduceLROnPlateau(factor=0.5, patience=2, monitor='val_loss', min_lr=1e-7)
]

history2 = model.fit(
    train_data,
    validation_data=test_data,
    epochs=15,
    callbacks=callbacks2,
    class_weight=class_weights
)
print(f'Phase 2 Best: {max(history2.history["val_accuracy"])*100:.2f}%')

In [ ]:
# Evaluate
loss, acc = model.evaluate(test_data)
print(f'Test Accuracy: {acc*100:.2f}%')

preds        = (model.predict(test_data) > 0.5).astype(int).flatten()
true_classes = test_data.classes
class_names  = list(test_data.class_indices.keys())

print('\nConfusion Matrix:')
print(confusion_matrix(true_classes, preds))
print('\nClassification Report:')
print(classification_report(true_classes, preds, target_names=class_names))

In [ ]:
save_dir = '/content/drive/MyDrive/ml_models'
os.makedirs(save_dir, exist_ok=True)
model.save(f'{save_dir}/skin_model.h5')
print('✅ skin_model.h5 saved!')
print('Classes:', train_data.class_indices)